# 02 — Parametric Geometry Explorer

**MANTA** — BWB Flying Wing with integrated center body

This notebook visualizes the 32-variable parametric model:
1. Default BWB configuration (2-wing AeroSandbox model)
2. Center-body airfoil sections (Kulfan CST)
3. Outer wing Kulfan airfoils
4. C0 continuity verification at the blend station
5. Internal volume computation
6. EDF duct fit verification
7. Design space bounds overview

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from src.parameterization.design_variables import (
    BWBParams, BOUNDS, VAR_NAMES, N_VARS, is_duct_compatible,
)
from src.parameterization.bwb_aircraft import (
    build_airplane, build_body_kulfan_at_station, build_kulfan_airfoil_at_station,
    compute_wing_area, compute_aspect_ratio, compute_mac,
    compute_internal_volume, estimate_structural_mass,
    body_height_at_xc,
    OUTER_WING_STATIONS, N_OUTER_SEGMENTS, BODY_STATIONS,
)
from src.propulsion.edf_model import EDF_70MM
from src.propulsion.duct_geometry import (
    size_and_place_duct, validate_duct_clearance,
    compute_duct_centerline,
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from src.visualization.style import apply_style, COLORS
apply_style()

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Default Configuration

In [ ]:
from src.config import load_all, duct_from_config, load_config
cfg = load_all()
mission = cfg['mission']
duct_cfg = duct_from_config(load_config())

params = BWBParams()

print(f"=== Mission (from config/default.yaml) ===")
print(f"  V_cruise  : {mission.velocity} m/s ({mission.velocity*3.6:.0f} km/h)")
print(f"  MTOW      : {mission.mtow} kg")
print(f"  Payload   : {mission.payload_mass*1000:.0f}g")
print()

print(f"=== Design Variables ({N_VARS} total) ===")
print(f"\nPlanform:")
print(f"  Half span        : {params.half_span:.3f} m  (full: {2*params.half_span:.2f} m)")
print(f"  Wing root chord  : {params.wing_root_chord:.3f} m")
print(f"  Taper ratio      : {params.taper_ratio}")
print(f"  LE sweep         : {params.le_sweep_deg} deg")
print(f"\nCenter Body:")
print(f"  Body chord ratio : {params.body_chord_ratio} -> chord = {params.body_root_chord:.3f} m")
print(f"  Body half-width  : {params.body_halfwidth:.3f} m")
print(f"  Body Kulfan u2   : {params.body_kulfan_u2:.3f}")
print(f"  Body Kulfan u6   : {params.body_kulfan_u6:.3f}")
print(f"  Body Kulfan u7   : {params.body_kulfan_u7:.3f}")
print(f"  Body Kulfan l2   : {params.body_kulfan_l2:.3f}")
print(f"  Body Kulfan l6   : {params.body_kulfan_l6:.3f}")
print(f"  Body Kulfan l7   : {params.body_kulfan_l7:.3f}")
print(f"  Body t/c (derived): {params.body_tc_root:.1%}")
print(f"  Body twist       : {params.body_twist} deg")
print(f"  Body sweep       : {params.body_sweep_deg} deg (wing + {params.body_sweep_delta} deg)")
print(f"\nDerived Geometry:")
print(f"  Outer half-span  : {params.outer_half_span:.3f} m")
print(f"  Tip chord        : {params.tip_chord:.3f} m")
s_ref = compute_wing_area(params)
print(f"  Wing area        : {s_ref:.4f} m2")
print(f"  Aspect ratio     : {compute_aspect_ratio(params):.2f}")
print(f"  MAC              : {compute_mac(params):.3f} m")
print(f"  Charge alaire    : {mission.mtow / s_ref:.1f} kg/m2")
print(f"  Struct mass      : {estimate_structural_mass(params, mtow=mission.mtow):.3f} kg")
print(f"  Internal volume  : {compute_internal_volume(params)*1e6:.0f} cm3")
print(f"  Duct compatible  : {is_duct_compatible(params)}")

## 2. Planform — Top View & Front View

The BWB has two distinct regions: center body (thick, more swept) and outer wing (tapered, Kulfan profiles).

In [ ]:
p = params
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Compute body positions ──
body_sweep_rad = np.radians(p.body_sweep_deg)
bw = p.body_halfwidth
body_chord_root = p.body_root_chord
body_chord_tip = p.wing_root_chord

body_le = [[0, 0], [bw * 0.5 * np.tan(body_sweep_rad), bw * 0.5],
           [bw * np.tan(body_sweep_rad), bw]]
body_te = [[body_chord_root, 0],
           [(body_chord_root + body_chord_tip) / 2 + bw * 0.5 * np.tan(body_sweep_rad), bw * 0.5],
           [body_chord_tip + bw * np.tan(body_sweep_rad), bw]]

# ── Compute outer wing positions ──
wing_sweep_rad = np.radians(p.le_sweep_deg)
outer_span = p.outer_half_span
root_chord = p.wing_root_chord
tip_chord = p.tip_chord
dihedrals = [p.dihedral_0, p.dihedral_1, p.dihedral_2, p.dihedral_2, p.dihedral_3, p.dihedral_tip]

x_blend = bw * np.tan(body_sweep_rad)
wing_le = [[x_blend, bw, 0]]
for i in range(N_OUTER_SEGMENTS):
    prev = wing_le[-1]
    dy_seg = outer_span * (OUTER_WING_STATIONS[i+1] - OUTER_WING_STATIONS[i])
    dih_rad = np.radians(dihedrals[i])
    dx = dy_seg * np.tan(wing_sweep_rad)
    dy = dy_seg * np.cos(dih_rad)
    dz = dy_seg * np.sin(dih_rad)
    wing_le.append([prev[0]+dx, prev[1]+dy, prev[2]+dz])

chords = [root_chord + f*(tip_chord-root_chord) for f in OUTER_WING_STATIONS]
wing_te = [[le[0]+c, le[1], le[2] if len(le)>2 else 0] for le, c in zip(wing_le, chords)]

# ═══ TOP VIEW ═══
ax = axes[0]
for sign in [1, -1]:
    # Body
    by = [pt[1]*sign for pt in body_le]
    bx_le = [pt[0] for pt in body_le]
    bx_te = [pt[0] for pt in body_te]
    ax.fill_betweenx(by, bx_le, bx_te, alpha=0.15, color=COLORS['body'])
    ax.plot(bx_le, by, color=COLORS['body'], lw=2)
    ax.plot(bx_te, by, color=COLORS['body'], lw=2)

    # Outer wing
    wy = [pt[1]*sign for pt in wing_le]
    wx_le = [pt[0] for pt in wing_le]
    wx_te = [pt[0] for pt in wing_te]
    ax.fill_betweenx(wy, wx_le, wx_te, alpha=0.1, color=COLORS['wing'])
    ax.plot(wx_le, wy, color=COLORS['wing'], lw=2)
    ax.plot(wx_te, wy, color=COLORS['wing'], lw=2)
    ax.plot([wx_le[-1], wx_te[-1]], [wy[-1], wy[-1]], color=COLORS['wing'], lw=2)

# Root closure
ax.plot([0, body_chord_root], [0, 0], color=COLORS['body'], lw=2)
ax.plot(p.body_root_chord * 0.30, 0, 'o', color=COLORS['feasible'], ms=8, label="CG ref")
ax.set_xlabel("Chord [m]")
ax.set_ylabel("Span [m]")
ax.set_title(f"Top View — span={2*p.half_span:.2f}m, AR={compute_aspect_ratio(p):.1f}")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.2)

# ═══ FRONT VIEW ═══
ax2 = axes[1]
for sign in [1, -1]:
    ax2.plot([0, bw*sign], [0, 0], color=COLORS['body'], lw=3,
             label="Body" if sign==1 else None)
    wy = [pt[1]*sign for pt in wing_le]
    wz = [pt[2] if len(pt)>2 else 0 for pt in wing_le]
    ax2.plot(wy, wz, color=COLORS['wing'], lw=2.5,
             label="Wing" if sign==1 else None)

ax2.plot(0, 0, 'o', color=COLORS['feasible'], ms=8)
ax2.set_xlabel("Span [m]")
ax2.set_ylabel("Height [m]")
ax2.set_title(f"Front View — Gull Wing [{p.dihedral_0:.0f}°, {p.dihedral_1:.0f}°, "
              f"{p.dihedral_2:.0f}°, {p.dihedral_3:.0f}°, {p.dihedral_tip:.0f}°]")
ax2.set_aspect("equal")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2b. Control Surfaces — Elevons & Ailerons

Materialized hinge lines and flap outlines on the outer wing. Computed from `ControlConfig.default_bwb()`:
- **Elevon** (pitch): 25% chord, η=0.00→0.50 (symmetric)
- **Aileron** (roll): 25% chord, η=0.55→0.90 (antisymmetric)

In [ ]:
from src.geometry.control_surfaces import (
    compute_control_surface_geometry, print_control_surface_summary,
    compute_wing_le_positions,
)
from src.config import load_all

cfg = load_all()
mission = cfg['mission']
controls = cfg['controls']

geoms = compute_control_surface_geometry(params, controls, n_spanwise=30)

# ── Print manufacturing summary ──
print_control_surface_summary(geoms)

# ── Top view with control surfaces ──
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

# Wing planform
body_sweep_rad = np.radians(p.body_sweep_deg)
bw = p.body_halfwidth
x_blend = bw * np.tan(body_sweep_rad)

body_le = [0, x_blend]
body_te = [p.body_root_chord, x_blend + p.wing_root_chord]
body_y = [0, bw]

le_xyz, chords, _ = compute_wing_le_positions(params)
wing_le_x = le_xyz[:, 0]
wing_te_x = le_xyz[:, 0] + chords
wing_y = le_xyz[:, 1]

outline_x = np.concatenate([body_le, wing_le_x[1:], wing_te_x[::-1], body_te[::-1], [body_le[0]]])
outline_y = np.concatenate([body_y, wing_y[1:], wing_y[::-1], body_y[::-1], [body_y[0]]])

ax.fill(outline_x, outline_y, alpha=0.08, color='#2c3e50')
ax.fill(outline_x, -outline_y, alpha=0.08, color='#2c3e50')
ax.plot(outline_x, outline_y, 'k-', linewidth=1.5)
ax.plot(outline_x, -outline_y, 'k-', linewidth=1.5)

# Control surfaces
cs_colors = {'elevon': '#e67e22', 'aileron': '#3498db'}
for g in geoms:
    color = cs_colors.get(g.name, 'gray')
    fill_x = np.concatenate([g.hinge_line_upper[:, 0], g.te_line_upper[::-1, 0]])
    fill_y = np.concatenate([g.hinge_line_upper[:, 1], g.te_line_upper[::-1, 1]])
    ax.fill(fill_x, fill_y, alpha=0.35, color=color, label=f'{g.name} ({g.span*100:.0f}cm, {g.area*1e4:.0f}cm2)')
    ax.fill(fill_x, -fill_y, alpha=0.35, color=color)

    ax.plot(g.hinge_line_upper[:, 0], g.hinge_line_upper[:, 1], '--', color=color, linewidth=2)
    ax.plot(g.hinge_line_upper[:, 0], -g.hinge_line_upper[:, 1], '--', color=color, linewidth=2)

    for idx in [0, -1]:
        x_bnd = [g.hinge_line_upper[idx, 0], g.te_line_upper[idx, 0]]
        y_bnd = [g.hinge_line_upper[idx, 1], g.te_line_upper[idx, 1]]
        ax.plot(x_bnd, y_bnd, '-', color=color, linewidth=1.5, alpha=0.7)
        ax.plot(x_bnd, [-y for y in y_bnd], '-', color=color, linewidth=1.5, alpha=0.7)

# CG marker
from src.systems.cg import compute_cg
cg = compute_cg(params, mission.battery_mass, mission.motor_mass, mission.avionics_mass, cfg['cg'])
ax.plot(cg['x_cg'], 0, 'go', markersize=12, zorder=10, label=f'CG (x/c={cg["x_cg_frac"]:.2f})')

ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=9)
s_ref = compute_wing_area(params)
ax.set_title(f'Top View  |  span={2*p.half_span:.2f}m  AR={4*p.half_span**2/s_ref:.1f}  S={s_ref*1e4:.0f}cm2',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Airfoil Sections

4 key sections: body center (thick), body blend/wing root, wing mid-span, wing tip.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)

sections = [
    ("Body Center (y=0)", build_body_kulfan_at_station(p, 0.0, "center").to_airfoil(), "coral"),
    ("Wing Root / Body Blend", build_kulfan_airfoil_at_station(p, 0.0, "root").to_airfoil(), "steelblue"),
    ("Wing 50% span", build_kulfan_airfoil_at_station(p, 0.5, "mid").to_airfoil(), "steelblue"),
    ("Wing Tip", build_kulfan_airfoil_at_station(p, 1.0, "tip").to_airfoil(), "steelblue"),
]

for ax, (title, af, color) in zip(axes.flat, sections):
    coords = af.coordinates
    ax.plot(coords[:, 0], coords[:, 1], color=color, lw=1.5)
    ax.fill(coords[:, 0], coords[:, 1], alpha=0.1, color=color)
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("y/c")

axes[1, 0].set_xlabel("x/c")
axes[1, 1].set_xlabel("x/c")

plt.tight_layout()
plt.show()

## 4. AeroSandbox Model — Build & Verify

Build the full 2-wing AeroSandbox model and verify the structure.

In [ ]:
airplane = build_airplane(params)

print(f"Airplane: {airplane.name}")
print(f"Wings: {len(airplane.wings)}")
for wing in airplane.wings:
    print(f"\n  {wing.name}:")
    print(f"    Symmetric: {wing.symmetric}")
    print(f"    Sections: {len(wing.xsecs)}")
    for j, xsec in enumerate(wing.xsecs):
        print(f"      [{j}] LE={[f'{v:.4f}' for v in xsec.xyz_le]}, "
              f"chord={xsec.chord:.4f}m, twist={xsec.twist:.1f}°, "
              f"airfoil={xsec.airfoil.name}")

# C0 continuity check
body_tip = airplane.wings[0].xsecs[-1]  # last body section
wing_root = airplane.wings[1].xsecs[0]  # first wing section
print(f"\n═══ C0 Continuity at Blend ═══")
print(f"  Body tip LE  : {body_tip.xyz_le}")
print(f"  Wing root LE : {wing_root.xyz_le}")
print(f"  LE match     : {np.allclose(body_tip.xyz_le, wing_root.xyz_le, atol=1e-6)}")
print(f"  Chord match  : body={body_tip.chord:.4f}, wing={wing_root.chord:.4f} → "
      f"{'OK' if abs(body_tip.chord - wing_root.chord) < 1e-6 else 'MISMATCH'}")

## 5. Design Space Bounds

In [ ]:
print(f"{'Variable':<28s} {'Lower':>8s} {'Default':>8s} {'Upper':>8s}")
print("─" * 56)
d = {k: getattr(params, k) for k in VAR_NAMES}
for name in VAR_NAMES:
    lo, hi = BOUNDS[name]
    val = d[name]
    print(f"{name:<28s} {lo:8.3f} {val:8.3f} {hi:8.3f}")

## 6. EDF Integration — Duct Placement & Clearance

The EDF 70mm (duct OD = 78mm) must fit inside the center body.
Duct placement is computed from the body Kulfan CST profile:
- **Intake** (x/c ≈ 0.04): flush NACA scoop on dorsal surface
- **Fan** (x/c ≈ 0.40): adaptive placement at maximum body height
- **Exhaust** (x/c ≈ 0.93): convergent nozzle at trailing edge

In [ ]:
# ── EDF Integration: placement, clearance, XZ view ──
placement = size_and_place_duct(params, EDF_70MM, config=duct_cfg)
all_ok, clr_results = validate_duct_clearance(placement, params, min_clearance_mm=0.0)

print(f"EDF: {EDF_70MM.name} (duct OD = {EDF_70MM.duct_outer_diameter*1000:.0f} mm)")
print(f"Body chord: {params.body_root_chord*1000:.0f} mm")
print(f"Body height at fan (x/c=0.40): {body_height_at_xc(params, 0.40) * params.body_root_chord * 1000:.1f} mm")
print(f"Fan at x/c={placement.fan_x_frac:.2f} ({placement.fan_x*1000:.0f} mm)")
min_clr = min(min(r.clearance_top_mm, r.clearance_bot_mm) for r in clr_results)
print(f"Min clearance: {min_clr:.1f} mm | Duct fits: {all_ok}")

# ── XZ integration view ──
bc = params.body_root_chord
af = build_body_kulfan_at_station(params, 0.0).to_airfoil()
af_x = af.coordinates[:, 0] * bc * 1000
af_z = af.coordinates[:, 1] * bc * 1000
centerline = compute_duct_centerline(placement, n_pts=100)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[3, 1])

ax1.fill_between(af_x, af_z, alpha=0.1, color='#2c3e50', label='Body envelope')
ax1.plot(af_x, af_z, 'k-', linewidth=2)
ax1.plot(centerline[:, 0]*1000, centerline[:, 2]*1000, 'r-', linewidth=2.5, label='Duct centerline')
ax1.plot(placement.intake_x*1000, placement.intake_z*1000, 'go', ms=10, label='Intake')
ax1.plot(placement.fan_x*1000, placement.fan_z*1000, 'bs', ms=10, label='Fan face')
ax1.plot(placement.exhaust_x*1000, placement.exhaust_z*1000, 'r^', ms=10, label='Exhaust')

fan_r = placement.duct_od / 2 * 1000
fan_circle = Circle((placement.fan_x*1000, placement.fan_z*1000), fan_r,
                     fill=False, color='#3498db', linewidth=2, label='EDF outline')
ax1.add_patch(fan_circle)
ax1.set_ylabel('Z [mm]')
ax1.set_title('Duct Integration — Longitudinal XZ View (y=0)', fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.2)

clr_x = [r.x_mm for r in clr_results]
clr_top = [r.clearance_top_mm for r in clr_results]
clr_bot = [r.clearance_bot_mm for r in clr_results]
ax2.plot(clr_x, clr_top, 'b-', lw=2, label='Top clearance')
ax2.plot(clr_x, clr_bot, 'r-', lw=2, label='Bottom clearance')
ax2.axhline(0, color='k', lw=0.5)
ax2.axhline(5.0, color='k', ls='--', lw=1, alpha=0.5, label='Min 5mm')
ax2.fill_between(clr_x, clr_bot, 0, where=[b < 0 for b in clr_bot], alpha=0.3, color='red')
ax2.fill_between(clr_x, clr_top, 0, where=[t < 0 for t in clr_top], alpha=0.3, color='orange')
ax2.set_xlabel('X [mm]')
ax2.set_ylabel('Clearance [mm]')
ax2.set_title('Duct-to-OML Clearance Along Centerline')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()